In [ ]:
import pandas as pd,numpy as np,os,pickle
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer,util
from scipy.sparse import csr_matrix
from warnings import filterwarnings
from dotenv import load_dotenv
load_dotenv()
filterwarnings('ignore')
os.environ['HF_TOKEN']=os.getenv('HF_API_KEY')

In [170]:
df=pd.read_csv('Employees Job Desc\job_dataset.csv')
df.head()

,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords
0,NET-F-001,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,Assist in coding and debugging applications; L...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
1,NET-F-002,.NET Developer,Fresher,0-1,C#; .NET Framework basics; ASP.NET; Razor; HTM...,Write simple C# programs under guidance; Suppo...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
2,NET-F-003,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Core; ASP.NET MVC; HTM...,Contribute to development of small modules; As...,.NET; C#; ASP.NET MVC; SQL Server; Entity Fram...
3,NET-F-004,.NET Developer,Fresher,0-1,C#; .NET Framework; ASP.NET basics; SQL Server...,Support in software design documentation; Assi...,.NET; C#; SQL Server; Entity Framework; ASP.NET
4,NET-F-005,.NET Developer,Fresher,0-1,C#; ASP.NET; MVC; Entity Framework basics; SQL...,Learn to design and build ASP.NET applications...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...


In [171]:
df.isna().sum()

JobID                0
Title                1
ExperienceLevel      0
YearsOfExperience    0
Skills               0
Responsibilities     0
Keywords             0
dtype: int64

In [172]:
df[df.Title.isna()]

,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords
36,AI017,NaN,Senior-Level,5–10 years,Python; Java; C++; R; Deep Learning; NLP; Comp...,Lead complex AI and ML projects; Develop scala...,AI; ML; Senior-Level; Deep Learning; NLP; Clou...


In [173]:
df.loc[df.Title.isna(),'Title']='Machine Learning Engineer'

In [174]:
df.Title.str.contains('Machine Learning Engineer',case=False).sum()

np.int64(9)

In [175]:
df.columns

Index(['JobID', 'Title', 'ExperienceLevel', 'YearsOfExperience', 'Skills',
       'Responsibilities', 'Keywords'],
      dtype='object')

In [176]:
df.Responsibilities.unique()

array(['Assist in coding and debugging applications; Learn and apply .NET Framework and Core fundamentals; Support team in building ASP.NET MVC web applications; Write basic SQL queries and work with Entity Framework; Collaborate with peers to solve issues; Participate in code reviews for learning; Follow best practices in coding; Work with version control (Git)',
       'Write simple C# programs under guidance; Support development of ASP.NET MVC applications; Implement Razor views and front-end logic; Assist in database query writing; Participate in unit testing tasks; Learn and apply LINQ for data operations; Work with mentors for code corrections',
       'Contribute to development of small modules; Assist in bug fixing and debugging; Learn and implement MVC patterns; Support database integration tasks; Understand version control basics; Work on minor testing scripts; Follow coding standards',
       ...,
       'Lead Vue.js front-end development; Integrate Node.js back-end services

In [177]:
pd.crosstab(df['ExperienceLevel'],df['YearsOfExperience'])

YearsOfExperience,0,0-1,0-1 Years,0-2,0–1 year,1-2 years,10,10+,10+ years,10-14,10-15,11+,11-15,12,12+,12-15,12-16,13+,13-15,13-16,13-17,14+,14-18,15+,15-20,16+,16-20,17+,18+,18-20,1–3 years,2,2 years,2-3,2-3 years,2-4,2-4 Years,2-5,2-5 Years,2–4 years,...,5+,5+ Years,5+ years,5-10,5-7,5-8,5-9,5–10 years,5–7,5–7 years,5–8 years,5–9 years,6,6+,6+ years,6-10,6-12,6-8,6-9,6–10 years,6–8,6–8 years,7,7+,7+ years,7-10,7-12,7-9,7–10,7–10 years,7–9 years,8,8+,8-10,8-12,8–10 years,9,9+,9-12,9–12 years
ExperienceLevel,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Entry-Level,0,0,0,0,66,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Experienced,0,0,0,0,0,1,1,12,2,1,3,7,1,1,7,3,1,6,1,1,1,4,2,10,4,4,1,3,1,1,1,4,1,2,1,1,1,0,1,4,...,43,2,13,25,15,21,2,0,2,1,0,0,2,18,1,13,7,6,6,0,2,1,3,10,0,9,2,1,2,0,1,2,11,4,2,1,2,8,3,1
Fresher,48,242,10,25,38,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Junior,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Lead,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,3,0,0,0,0,0,0,0
Mid-Level,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,35,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Mid-Senior,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Mid-Senior Level,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Mid-level,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [178]:
df.YearsOfExperience.value_counts()

YearsOfExperience
0-1           247
0–1 year      104
5+             49
0              48
2-5            35
             ... 
5–7 years       1
1–3 years       1
13-17           1
7–10 years      1
7-9             1
Name: count, Length: 110, dtype: int64

In [179]:
df['YearsOfExperience']=df['YearsOfExperience'].apply(lambda x:x.split(' ')[0].strip() if 'year' in x.lower() else x.lower().strip())
df['YearsOfExperience'].value_counts()

YearsOfExperience
0-1      257
0–1      104
5+        71
0         48
6+        36
        ... 
13-17      1
7–9        1
8–10       1
9–12       1
7-9        1
Name: count, Length: 84, dtype: int64

In [180]:
df['YearsOfExperience']=df['YearsOfExperience'].str.extract(r'(\d+)\+?$|\D*(\d+)$').bfill(axis=1)[0].astype(int)
df['YearsOfExperience'].value_counts()

YearsOfExperience
1     361
5     155
10     81
6      79
8      58
7      54
0      48
4      47
3      41
9      35
2      31
12     23
15     18
11      7
16      6
13      6
20      6
14      5
17      4
18      3
Name: count, dtype: int64

In [181]:
pd.crosstab(df['ExperienceLevel'],df['YearsOfExperience'])

YearsOfExperience,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,20
ExperienceLevel,,,,,,,,,,,,,,,,,,,,
Entry-Level,0,66,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Experienced,0,0,6,41,42,98,43,34,45,20,69,7,23,6,5,18,6,4,3,6
Fresher,48,290,25,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Junior,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Lead,0,0,0,0,0,0,0,4,3,0,0,0,0,0,0,0,0,0,0,0
Mid-Level,0,0,0,0,0,44,13,3,0,0,0,0,0,0,0,0,0,0,0,0
Mid-Senior,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0
Mid-Senior Level,0,0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0,0,0
Mid-level,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [182]:
df.Responsibilities[0]

'Assist in coding and debugging applications; Learn and apply .NET Framework and Core fundamentals; Support team in building ASP.NET MVC web applications; Write basic SQL queries and work with Entity Framework; Collaborate with peers to solve issues; Participate in code reviews for learning; Follow best practices in coding; Work with version control (Git)'

In [183]:
def tfidf_vectorizer(df):
    tfidf=TfidfVectorizer(stop_words='english')
    df['Combined_Text']=df['Title']+' '+df['Skills']+ ' '+df['Responsibilities']+' '+df['Keywords']
    tfidf_df=tfidf.fit_transform(df['Combined_Text'])
    return tfidf_df,tfidf
def tfidf_get_recommendations(job_title,req_skills,min_exp,df,n=5):
    filtered_df=df[df['YearsOfExperience']>=min_exp]
    if filtered_df.empty:
        return pd.DataFrame(columns=['Title','Skills','Responsibilities','Keywords','YearsOfExperience','Cosine_Score'])
    tfidf_df,tfidf=tfidf_vectorizer(filtered_df)
    request_text=f'{job_title} {req_skills}'
    request_tfidf=tfidf.transform([request_text])
    cos_scores=cosine_similarity(request_tfidf,tfidf_df).flatten()
    filtered_df['Cosine_Score']=cos_scores
    sorted_df=filtered_df.sort_values(by='Cosine_Score',ascending=False)
    return sorted_df[['Title','Skills','Responsibilities','Keywords','YearsOfExperience','Cosine_Score']].head(n)
tfidf_get_recommendations('Data Scientist','Python,SQL,JAVA, Machine Learning',3,df,n=5)

,Title,Skills,Responsibilities,Keywords,YearsOfExperience,Cosine_Score
349,Data Scientist - Experienced,Python (advanced); R; SQL; Java/Scala; Regress...,Lead data science projects; Develop predictive...,Data Scientist; Machine Learning; Python; R; S...,6,0.594711
352,Principal Data Scientist,Python; R; SQL; Java/Scala; Machine Learning; ...,Lead strategic analytics projects; Develop adv...,Data Science; AI; Machine Learning; Big Data; ...,7,0.522903
353,Machine Learning Scientist,Python; R; SQL; Java/Scala; Regression; Classi...,Build advanced predictive models; Optimize ML ...,Machine Learning; Python; R; SQL; AI; Deep Lea...,5,0.496263
355,Lead Machine Learning Scientist,Python; R; SQL; Java/Scala; Ensemble methods; ...,Lead ML/AI projects end-to-end; Develop advanc...,Machine Learning; AI; Python; R; SQL; Big Data...,6,0.474289
350,Senior Data Scientist,Python; R; SQL; Java/Scala; Ensemble methods; ...,Architect and deploy advanced analytics and ML...,Data Science; Machine Learning; Python; R; SQL...,5,0.407134


In [184]:
df.head()

,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords
0,NET-F-001,.NET Developer,Fresher,1,C#; VB.NET basics; .NET Framework; .NET Core f...,Assist in coding and debugging applications; L...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
1,NET-F-002,.NET Developer,Fresher,1,C#; .NET Framework basics; ASP.NET; Razor; HTM...,Write simple C# programs under guidance; Suppo...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
2,NET-F-003,.NET Developer,Fresher,1,C#; VB.NET basics; .NET Core; ASP.NET MVC; HTM...,Contribute to development of small modules; As...,.NET; C#; ASP.NET MVC; SQL Server; Entity Fram...
3,NET-F-004,.NET Developer,Fresher,1,C#; .NET Framework; ASP.NET basics; SQL Server...,Support in software design documentation; Assi...,.NET; C#; SQL Server; Entity Framework; ASP.NET
4,NET-F-005,.NET Developer,Fresher,1,C#; ASP.NET; MVC; Entity Framework basics; SQL...,Learn to design and build ASP.NET applications...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...


In [186]:
sent_model=SentenceTransformer('all-MiniLM-L6-v2')
def generate_embeddings(df,sent_model):
    df['Combined_Text']=df['Title']+' '+df['Skills']+ ' '+df['Responsibilities']+' '+df['Keywords']
    embeddings=sent_model.encode(df['Combined_Text'].tolist(),convert_to_tensor=True)
    return embeddings
def transformer_recommendations(job_title,req_skills,min_exp,df,sent_model,n=5):
    request_text=f'{job_title} {req_skills}'
    request_embedding=sent_model.encode(request_text,convert_to_tensor=True)
    embeddings=generate_embeddings(df,sent_model)
    cos_scores=util.cos_sim(request_embedding,embeddings).reshape(-1,1)
    df['Cosine_Score_Sentence_Transformer']=cos_scores.cpu().numpy()
    filtered_df=df[df['YearsOfExperience']>=min_exp]
    filtered_df=filtered_df.sort_values(by='Cosine_Score_Sentence_Transformer',ascending=False)
    return filtered_df[['Title','Skills','Responsibilities','Keywords','YearsOfExperience','Cosine_Score_Sentence_Transformer']].head(n)
transformer_recommendations('Data Scientist','Python,SQL,JAVA, Machine Learning',3,df,sent_model,n=5)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6034.63it/s]


,Title,Skills,Responsibilities,Keywords,YearsOfExperience,Cosine_Score_Sentence_Transformer
349,Data Scientist - Experienced,Python (advanced); R; SQL; Java/Scala; Regress...,Lead data science projects; Develop predictive...,Data Scientist; Machine Learning; Python; R; S...,6,0.742001
352,Principal Data Scientist,Python; R; SQL; Java/Scala; Machine Learning; ...,Lead strategic analytics projects; Develop adv...,Data Science; AI; Machine Learning; Big Data; ...,7,0.735376
356,Data Science Team Lead,Python; R; SQL; Java/Scala; Classification; Re...,Manage a team of data scientists; Oversee mode...,Data Science; AI; Machine Learning; Big Data; ...,7,0.711222
350,Senior Data Scientist,Python; R; SQL; Java/Scala; Ensemble methods; ...,Architect and deploy advanced analytics and ML...,Data Science; Machine Learning; Python; R; SQL...,5,0.703446
351,Lead Data Scientist,Python; R; SQL; Java/Scala; Classification; Re...,Drive end-to-end data science initiatives; Imp...,Data Science; AI; Machine Learning; Python; R;...,6,0.693424


In [ ]:
df['Transformer_Embeddings'] = list(generate_embeddings(df, sent_model).cpu().numpy())
tfidf_matrix, trained_tfidf = tfidf_vectorizer(df)
session_data = {
    'dataframe': df,
    'tfidf_vectorizer': trained_tfidf,
    'tfidf_matrix': tfidf_matrix
    }
with open('search_assets.pkl', 'wb') as f:
    pickle.dump(session_data, f)
print("Assets saved successfully!")

Assets saved successfully!


In [187]:
with open('search_assets.pkl', 'rb') as f:
    assets=pickle.load(f)
loaded_df=assets['dataframe']
trained_tfidf=assets['tfidf_vectorizer']
tfidf_matrix=assets['tfidf_matrix']
query='Data Scientist Python SQL JAVA Machine Learning'
query_vector=trained_tfidf.transform([query])
cosine_similarities=cosine_similarity(query_vector,tfidf_matrix).flatten()

In [188]:
cosine_similarities_df=pd.DataFrame({'Index':range(len(cosine_similarities)),'Cosine_Similarity':cosine_similarities})
cosine_similarities_df=cosine_similarities_df.sort_values(by='Cosine_Similarity',ascending=False)
cosine_similarities_df=pd.merge(cosine_similarities_df,loaded_df,left_on='Index',right_index=True)
if cosine_similarities_df.empty:
    print("No matching job descriptions found.")
cosine_similarities_df=cosine_similarities_df[cosine_similarities_df['YearsOfExperience']>=3]
cosine_similarities_df=cosine_similarities_df[['Title','Skills','Responsibilities','Keywords','YearsOfExperience','Cosine_Similarity']]
cosine_similarities_df.head()

,Title,Skills,Responsibilities,Keywords,YearsOfExperience,Cosine_Similarity
349,Data Scientist - Experienced,Python (advanced); R; SQL; Java/Scala; Regress...,Lead data science projects; Develop predictive...,Data Scientist; Machine Learning; Python; R; S...,6,0.594726
352,Principal Data Scientist,Python; R; SQL; Java/Scala; Machine Learning; ...,Lead strategic analytics projects; Develop adv...,Data Science; AI; Machine Learning; Big Data; ...,7,0.512933
353,Machine Learning Scientist,Python; R; SQL; Java/Scala; Regression; Classi...,Build advanced predictive models; Optimize ML ...,Machine Learning; Python; R; SQL; AI; Deep Lea...,5,0.498019
355,Lead Machine Learning Scientist,Python; R; SQL; Java/Scala; Ensemble methods; ...,Lead ML/AI projects end-to-end; Develop advanc...,Machine Learning; AI; Python; R; SQL; Big Data...,6,0.465896
350,Senior Data Scientist,Python; R; SQL; Java/Scala; Ensemble methods; ...,Architect and deploy advanced analytics and ML...,Data Science; Machine Learning; Python; R; SQL...,5,0.395272


In [196]:
print(transformer_recommendations('Data Scientist','Python,SQL,JAVA, Machine Learning',3,loaded_df,sent_model,n=5))
%timeit

                            Title  ... Cosine_Score_Sentence_Transformer
349  Data Scientist - Experienced  ...                          0.742001
352      Principal Data Scientist  ...                          0.735376
356        Data Science Team Lead  ...                          0.711222
350         Senior Data Scientist  ...                          0.703446
351           Lead Data Scientist  ...                          0.693424

[5 rows x 6 columns]


In [ ]:
def get_semantic_recommendations(job_title, req_skills, min_exp, df, sent_model, n=5):
    filtered_df = df[df['YearsOfExperience'] >= min_exp]
    if filtered_df.empty:
        return pd.DataFrame(columns=['Title', 'Skills', 'Responsibilities', 'Keywords', 'YearsOfExperience', 'Transformer_Score'])
    request_text = f'{job_title} {req_skills}'
    query_embedding = sent_model.encode([request_text]).reshape(1, -1)
    dataset_embeddings = np.vstack(filtered_df['Transformer_Embeddings'].values)
    cos_scores = cosine_similarity(dataset_embeddings, query_embedding).flatten()
    filtered_df['Transformer_Score'] = cos_scores
    sorted_df = filtered_df.sort_values(by='Transformer_Score', ascending=False)
    
    return sorted_df[['Title', 'Skills', 'Responsibilities', 'Keywords', 'YearsOfExperience', 'Transformer_Score']].head(n)
print(get_semantic_recommendations('Data Scientist', 'Python, SQL, JAVA, Machine Learning', 3, loaded_df, sent_model, n=5))

                            Title  ... Transformer_Score
349  Data Scientist - Experienced  ...          0.742001
352      Principal Data Scientist  ...          0.735376
356        Data Science Team Lead  ...          0.711222
350         Senior Data Scientist  ...          0.703446
351           Lead Data Scientist  ...          0.693424

[5 rows x 6 columns]


In [210]:
request_text = 'Data Scientist Python, SQL, JAVA, Machine Learning'
query_embedding = sent_model.encode([request_text]).flatten().reshape(1, -1)
np.vstack(loaded_df['Transformer_Embeddings']).shape

(1068, 384)

In [211]:
print(query_embedding.shape)
print(query_embedding)

(1, 384)
[[-5.70228212e-02 -4.67442758e-02  7.83364754e-03  6.38439134e-02
  -3.82725149e-02 -1.50322929e-01  1.91070549e-02  1.39356125e-04
  -1.59663171e-01 -3.21988426e-02 -7.22165257e-02 -2.34680753e-02
   4.95439656e-02 -1.81102492e-02  6.37137564e-03  6.73172101e-02
  -8.95320401e-02  1.40379267e-02  2.25619096e-02 -1.13402240e-01
  -7.57243782e-02 -1.21908626e-02  1.83357280e-02 -2.35558432e-02
   8.98371115e-02  1.95953697e-02  6.82219192e-02  4.00113687e-02
  -6.35271594e-02 -7.11900648e-03 -9.69707519e-02 -2.04455629e-02
  -3.39813419e-02  2.14217082e-02 -6.10835925e-02 -2.10197680e-02
   1.99269168e-02  1.06468182e-02 -1.70992818e-02  1.50077287e-02
  -4.51910794e-02 -4.21433933e-02  8.73133540e-03 -1.64323987e-03
   3.81005020e-03  1.24749579e-02 -4.08114158e-02 -4.58165295e-02
  -3.10454750e-03  5.08274045e-03 -1.52959555e-01 -3.63845825e-02
  -2.18682438e-02 -3.30158994e-02 -2.51372973e-03  2.48559862e-02
   6.91573918e-02  1.16455099e-02 -3.16114537e-02 -1.02936447e-01
 